# R9 · Géométrie du plan, vecteurs et trigonométrie — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r9_geometrie_vecteurs/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r9_geometrie_vecteurs/03_notebook.ipynb`, créée par `python tools/start_chapter.py R9`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R9.1 à R9.10, R9.13, R9.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | R9.16 | Le vecteur entre deux points et sa norme · `vector`, `norm` | 🔨 | ★ | 15 |
|  | R9.17 | Produit scalaire et angle · `dot`, `angle_between` | 🔨 | ★ | 20 |
|  | R9.18 | La droite qui passe par deux points, et les radians · `line_through`, `deg_to_rad` | 🔨 | ★ | 15 |
|  | R9.19 | Les mêmes calculs avec NumPy (`np.linalg.norm`, `@`) | 📦 | ★ | 15 |
| B | R9.20 | Prédire : permuter, décaler ou retourner des coordonnées change-t-il la similarité cosinus ? | 🔮 | ★ | 10 |
|  | R9.21 | Dessiner des vecteurs, leur somme et un angle | 🎨 | ★★ | 20 |
|  | R9.22 | Le cercle trigonométrique : cos et sin d'un angle qui tourne | 🔬 | ★★ | 20 |
| C | R9.23 | Chasse à l'erreur : radians et degrés | 🐛 | ★ | 15 |
|  | R9.24 | Comparer deux flottants : `np.isclose` | 🛠️ | ★ | 10 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R9")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : distances, vecteurs, normes, droites, produit scalaire, radians, cosinus et sinus, angles, vecteurs de ℝ³ et ℝ⁴.
- Écrire les six fonctions de `mymath/plane.py` et les tester, puis refaire les calculs avec NumPy.
- Prédire l'effet d'un multiple sur la similarité cosinus, dessiner des vecteurs, explorer le cercle trigonométrique, chasser des confusions entre degrés et radians, comparer des flottants.

**Rappel express.** $AB = \sqrt{(x_B - x_A)^2 + (y_B - y_A)^2}$ ; $\overrightarrow{AB} = B - A$ ; $\lVert \vec{u} \rVert = \sqrt{x^2 + y^2}$ ; $\vec{u} \cdot \vec{v} = xx' + yy' = \lVert \vec{u} \rVert\,\lVert \vec{v} \rVert \cos\theta$. $ax + by + c = 0$ : $(-b ; a)$ directeur, $(a ; b)$ normal. $180° = \pi$ rad ; `math.cos` attend des radians.

## Partie 0 · Vérifier tes exercices papier (✏️ R9.1 à R9.10, 🔗 R9.13 et R9.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"5/9"` ;
- une valeur exacte entre guillemets, avec `sqrt` et `pi`, `"sqrt(5)/3"` ou `"2*pi/5"` ;
- des coordonnées, un vecteur ou une équation $[a, b, c]$ par une liste entre crochets, `[-1.25, 1]` ; vrai ou faux par `True` ou `False`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(5, 9) is accepted too, as "5/9"

**Ex R9.1 — Coordonnées et distance**

In [ ]:
# Ex R9.1: the values you found on paper
answer_R9_1a = ...  # a) a list
answer_R9_1b = ...  # b) an integer
answer_R9_1c = ...  # c) a number (1 decimal)
answer_R9_1d = ...  # d) a number (2 decimals)
answer_R9_1e = ...  # e) True or False
answer_R9_1f = ...  # f) a number (2 decimals)
answer_R9_1g = ...  # g) an irreducible fraction, as text
answer_R9_1h = ...  # h) a list

for letter, answer in zip("abcdefgh", [answer_R9_1a, answer_R9_1b, answer_R9_1c, answer_R9_1d, answer_R9_1e, answer_R9_1f, answer_R9_1g, answer_R9_1h]):
    wbm.check(f"R9.1{letter}", answer)

**Ex R9.2 — Vecteurs : somme, différence, produit par un nombre**

In [ ]:
# Ex R9.2: the values you found on paper
answer_R9_2a = ...  # a) a list
answer_R9_2b = ...  # b) a list
answer_R9_2c = ...  # c) a list
answer_R9_2d = ...  # d) a list
answer_R9_2e = ...  # e) a list
answer_R9_2f = ...  # f) a list
answer_R9_2g = ...  # g) True or False
answer_R9_2h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_R9_2a, answer_R9_2b, answer_R9_2c, answer_R9_2d, answer_R9_2e, answer_R9_2f, answer_R9_2g, answer_R9_2h]):
    wbm.check(f"R9.2{letter}", answer)

**Ex R9.3 — Normes (euclidienne, avec un aperçu de L1 et L∞) et vecteurs unitaires**

In [ ]:
# Ex R9.3: the values you found on paper
answer_R9_3a = ...  # a) a number (2 decimals)
answer_R9_3b = ...  # b) a list, in increasing order
answer_R9_3c = ...  # c) a number (2 decimals)
answer_R9_3d = ...  # d) a list (2 decimals)
answer_R9_3e = ...  # e) a number (1 decimal)
answer_R9_3f = ...  # f) an integer
answer_R9_3g = ...  # g) an integer
answer_R9_3h = ...  # h) a list (2 decimals)

for letter, answer in zip("abcdefgh", [answer_R9_3a, answer_R9_3b, answer_R9_3c, answer_R9_3d, answer_R9_3e, answer_R9_3f, answer_R9_3g, answer_R9_3h]):
    wbm.check(f"R9.3{letter}", answer)

**Ex R9.4 — Équations de droites**

In [ ]:
# Ex R9.4: the values you found on paper
answer_R9_4a = ...  # a) a number (1 decimal)
answer_R9_4b = ...  # b) a number
answer_R9_4c = ...  # c) a list
answer_R9_4d = ...  # d) a list [a, b, c]
answer_R9_4e = ...  # e) an irreducible fraction, as text
answer_R9_4f = ...  # f) a number
answer_R9_4g = ...  # g) a list (1 decimal)
answer_R9_4h = ...  # h) a list [a, b, c]

for letter, answer in zip("abcdefgh", [answer_R9_4a, answer_R9_4b, answer_R9_4c, answer_R9_4d, answer_R9_4e, answer_R9_4f, answer_R9_4g, answer_R9_4h]):
    wbm.check(f"R9.4{letter}", answer)

**Ex R9.5 — Le produit scalaire en coordonnées**

In [ ]:
# Ex R9.5: the values you found on paper
answer_R9_5a = ...  # a) an integer
answer_R9_5b = ...  # b) an integer
answer_R9_5c = ...  # c) an integer
answer_R9_5d = ...  # d) an integer
answer_R9_5e = ...  # e) an integer
answer_R9_5f = ...  # f) an integer
answer_R9_5g = ...  # g) a number
answer_R9_5h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_R9_5a, answer_R9_5b, answer_R9_5c, answer_R9_5d, answer_R9_5e, answer_R9_5f, answer_R9_5g, answer_R9_5h]):
    wbm.check(f"R9.5{letter}", answer)

**Ex R9.6 — Orthogonalité et projections simples**

In [ ]:
# Ex R9.6: the values you found on paper
answer_R9_6a = ...  # a) True or False
answer_R9_6b = ...  # b) a list
answer_R9_6c = ...  # c) a number (1 decimal)
answer_R9_6d = ...  # d) True or False
answer_R9_6e = ...  # e) a number (2 decimals)
answer_R9_6f = ...  # f) a list
answer_R9_6g = ...  # g) a list
answer_R9_6h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_R9_6a, answer_R9_6b, answer_R9_6c, answer_R9_6d, answer_R9_6e, answer_R9_6f, answer_R9_6g, answer_R9_6h]):
    wbm.check(f"R9.6{letter}", answer)

**Ex R9.7 — Degrés et radians, cosinus et sinus remarquables**

In [ ]:
# Ex R9.7: the values you found on paper
answer_R9_7a = ...  # a) an exact value, as text
answer_R9_7b = ...  # b) an integer
answer_R9_7c = ...  # c) a number (1 decimal)
answer_R9_7d = ...  # d) an exact value, as text
answer_R9_7e = ...  # e) an exact value, as text
answer_R9_7f = ...  # f) an exact value, as text
answer_R9_7g = ...  # g) an exact value, as text
answer_R9_7h = ...  # h) an exact value, as text

for letter, answer in zip("abcdefgh", [answer_R9_7a, answer_R9_7b, answer_R9_7c, answer_R9_7d, answer_R9_7e, answer_R9_7f, answer_R9_7g, answer_R9_7h]):
    wbm.check(f"R9.7{letter}", answer)

**Ex R9.8 — Les courbes de cos et de sin : période, symétries, et arccos sur [0, π]**

In [ ]:
# Ex R9.8: the values you found on paper
answer_R9_8a = ...  # a) an exact value, as text
answer_R9_8b = ...  # b) a list, in increasing order
answer_R9_8c = ...  # c) an integer
answer_R9_8d = ...  # d) an exact value, as text
answer_R9_8e = ...  # e) an exact value, as text
answer_R9_8f = ...  # f) True or False
answer_R9_8g = ...  # g) an integer
answer_R9_8h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R9_8a, answer_R9_8b, answer_R9_8c, answer_R9_8d, answer_R9_8e, answer_R9_8f, answer_R9_8g, answer_R9_8h]):
    wbm.check(f"R9.8{letter}", answer)

**Ex R9.9 — L'angle entre deux vecteurs**

In [ ]:
# Ex R9.9: the values you found on paper
answer_R9_9a = ...  # a) a number (4 decimals)
answer_R9_9b = ...  # b) a number (1 decimal)
answer_R9_9c = ...  # c) an integer
answer_R9_9d = ...  # d) an integer
answer_R9_9e = ...  # e) an exact value, as text
answer_R9_9f = ...  # f) a number (1 decimal)
answer_R9_9g = ...  # g) an integer
answer_R9_9h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_R9_9a, answer_R9_9b, answer_R9_9c, answer_R9_9d, answer_R9_9e, answer_R9_9f, answer_R9_9g, answer_R9_9h]):
    wbm.check(f"R9.9{letter}", answer)

**Ex R9.10 — Distances et produits scalaires dans ℝ³ et ℝ⁴**

In [ ]:
# Ex R9.10: the values you found on paper
answer_R9_10a = ...  # a) an integer
answer_R9_10b = ...  # b) an integer
answer_R9_10c = ...  # c) an integer
answer_R9_10d = ...  # d) a number (3 decimals)
answer_R9_10e = ...  # e) True or False
answer_R9_10f = ...  # f) a number (2 decimals)
answer_R9_10g = ...  # g) an integer
answer_R9_10h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_R9_10a, answer_R9_10b, answer_R9_10c, answer_R9_10d, answer_R9_10e, answer_R9_10f, answer_R9_10g, answer_R9_10h]):
    wbm.check(f"R9.10{letter}", answer)

**Ex R9.13 — Deux recettes, deux vecteurs d'ingrédients : la similarité cosinus**

In [ ]:
# Ex R9.13: the values you found on paper
answer_R9_13a = ...  # a) an integer
answer_R9_13b = ...  # b) a number (2 decimals)
answer_R9_13c = ...  # c) a number (2 decimals)
answer_R9_13d = ...  # d) a number (3 decimals)
answer_R9_13e = ...  # e) a number (3 decimals)
answer_R9_13f = ...  # f) a number (3 decimals)
answer_R9_13g = ...  # g) a letter, as text
answer_R9_13h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_R9_13a, answer_R9_13b, answer_R9_13c, answer_R9_13d, answer_R9_13e, answer_R9_13f, answer_R9_13g, answer_R9_13h]):
    wbm.check(f"R9.13{letter}", answer)

**Ex R9.14 — Un neurone trace une droite : w₁x₁ + w₂x₂ + b = 0**

In [ ]:
# Ex R9.14: the values you found on paper
answer_R9_14a = ...  # a) an integer
answer_R9_14b = ...  # b) an integer
answer_R9_14c = ...  # c) a list
answer_R9_14d = ...  # d) a list
answer_R9_14e = ...  # e) a list
answer_R9_14f = ...  # f) True or False
answer_R9_14g = ...  # g) a list
answer_R9_14h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R9_14a, answer_R9_14b, answer_R9_14c, answer_R9_14d, answer_R9_14e, answer_R9_14f, answer_R9_14g, answer_R9_14h]):
    wbm.check(f"R9.14{letter}", answer)

## Partie A · Ta librairie : vecteurs, norme, produit scalaire, angle, droites, radians

Fiche §2 à §8. Les six fonctions de `mymath/plane.py`, puis les mêmes calculs avec NumPy.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, drawn, run_pytest_group, verdict


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"


def normalized(equation):
    """The equation (a, b, c) divided by the first non-zero of a and b, rounded: one form for every equation of a line."""
    a, b, c = equation
    first = a if a != 0 else b
    return [round(x / first, 4) + 0.0 for x in (a, b, c)]

### Ex R9.16 — Le vecteur entre deux points et sa norme · `vector`, `norm` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer les coordonnées d'un vecteur et sa norme, dans n'importe quelle dimension.  
**Prérequis :** fiche §2, §3 et §8 · **mymath :** `plane.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme dans les chapitres précédents). Ouvre ta copie `mon_travail/mymath/plane.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Comme dans tous les chapitres R0 à R9, Python pur et le module `math` (importe-le en haut du fichier : `import math`) : ni NumPy, ni SymPy (un test le vérifie). Écris toi-même les sommes de produits et de carrés (plutôt que `math.dist` ou `math.hypot`, qui feraient le travail à ta place) : c'est l'exercice. Les tests comparent tes résultats à NumPy et à des calculs exacts avec `Fraction`. Une cellule qui ne s'arrête pas (l'étoile `[*]` reste affichée) : interromps-la (■, *Interrupt*), puis cherche une boucle `while` dont la condition reste toujours vraie.

Écris `vector(A, B)` puis `norm(u)` dans `mon_travail/mymath/plane.py` (fiche §2 et §3). Un point et un vecteur sont des tuples de coordonnées, de n'importe quelle longueur (§8). `vector` renvoie un tuple, arrivée moins départ, et lève une `ValueError` si les deux points n'ont pas le même nombre de coordonnées ; `norm` renvoie un flottant.

La vérification calcule :
- a) `vector((-1, 4), (6, -2))` ;
- b) `vector((1, 2, 3), (0, 0, 0))` ;
- c) le nom de l'erreur levée par `vector((1, 2), (1, 2, 3))` (avec `raised`, de la cellule des outils) ;
- d) la liste `[norm((7, -24)), norm((1, 1, 1, 1)), norm((0, 0))]` ;
- e) le type de `norm((7, -24))`, en texte (`"float"` pour un flottant) ;
- f) la distance entre les points $(0 ; 2)$ et $(20 ; 23)$, calculée avec tes deux fonctions.

Puis, dans ta copie : pourquoi `norm` n'a-t-elle besoin d'aucun cas particulier pour la dimension 1, où $\lVert (x) \rVert = \lvert x \rvert$ ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R9")  # reload your saved file

In [ ]:
with checked("R9.16a"):
    wbm.check("R9.16a", mymath.plane.vector((-1, 4), (6, -2)), computed=True)
with checked("R9.16b"):
    wbm.check("R9.16b", mymath.plane.vector((1, 2, 3), (0, 0, 0)), computed=True)
with checked("R9.16c"):
    wbm.check("R9.16c", raised(mymath.plane.vector, (1, 2), (1, 2, 3)), computed=True)
with checked("R9.16d"):
    wbm.check("R9.16d", [mymath.plane.norm((7, -24)), mymath.plane.norm((1, 1, 1, 1)), mymath.plane.norm((0, 0))],
              computed=True)
with checked("R9.16e"):
    wbm.check("R9.16e", type(mymath.plane.norm((7, -24))).__name__, computed=True)
with checked("R9.16f"):
    wbm.check("R9.16f", mymath.plane.norm(mymath.plane.vector((0, 2), (20, 23))), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r9_plane.py", "test_vector_ or test_norm_ or test_plane_uses")

### Ex R9.17 — Produit scalaire et angle · `dot`, `angle_between` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le produit scalaire et l'angle entre deux vecteurs, sans être piégé par les arrondis des flottants.  
**Prérequis :** fiche §5 et §7 ; Ex R9.16 · **mymath :** `plane.py` · **Parcours :** C, E

Écris `dot(u, v)` puis `angle_between(u, v)` (fiche §5 et §7). `dot` renvoie la somme des produits des coordonnées (un entier si les coordonnées sont entières) et lève une `ValueError` si les dimensions diffèrent. `angle_between` renvoie l'angle en radians, dans $[0 ; \pi]$, avec `math.acos` ; elle lève une `ValueError` si l'un des vecteurs est nul. Réutilise `dot` et `norm`.

⚠️ Avec des flottants, le cosinus calculé peut valoir 1,0000000000000002 pour deux vecteurs de même sens, ou −1,0000000000000002 pour deux vecteurs de sens contraires : `math.acos` lève alors une erreur. Ramène le quotient dans $[-1 ; 1]$ avant de l'appeler (`max(-1.0, min(1.0, cosine))`).

La vérification calcule :
- a) `dot((2, -3, 1), (4, 1, -5))` ;
- b) le type de `dot((1, 2), (3, 4))`, en texte ;
- c) le nom de l'erreur levée par `dot((1, 2), (1, 2, 3))` ;
- d) `angle_between((1, 0), (-1, 2))` (4 décimales) ;
- e) le nom de l'erreur levée par `angle_between((1, 6), (-2, -12))` (`"aucune"` si elle n'en lève pas), puis f) sa valeur (4 décimales) ;
- g) le nom de l'erreur levée par `angle_between((0, 0), (1, 2))` ;
- h) l'angle entre $(-2 ; 1)$ et $(3 ; 4)$, en degrés (`math.degrees`, 4 décimales).

Puis, dans ta copie : pourquoi le cosinus de e) sort-il de $[-1 ; 1]$, alors que les deux vecteurs sont exactement de sens contraires ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R9")  # reload your saved file

In [ ]:
with checked("R9.17a"):
    wbm.check("R9.17a", mymath.plane.dot((2, -3, 1), (4, 1, -5)), computed=True)
with checked("R9.17b"):
    wbm.check("R9.17b", type(mymath.plane.dot((1, 2), (3, 4))).__name__, computed=True)
with checked("R9.17c"):
    wbm.check("R9.17c", raised(mymath.plane.dot, (1, 2), (1, 2, 3)), computed=True)
with checked("R9.17d"):
    wbm.check("R9.17d", mymath.plane.angle_between((1, 0), (-1, 2)), computed=True)
with checked("R9.17e"):
    wbm.check("R9.17e", raised(mymath.plane.angle_between, (1, 6), (-2, -12)), computed=True)
with checked("R9.17f"):
    wbm.check("R9.17f", mymath.plane.angle_between((1, 6), (-2, -12)), computed=True)
with checked("R9.17g"):
    wbm.check("R9.17g", raised(mymath.plane.angle_between, (0, 0), (1, 2)), computed=True)
with checked("R9.17h"):
    wbm.check("R9.17h", math.degrees(mymath.plane.angle_between((-2, 1), (3, 4))), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r9_plane.py", "test_dot_ or test_angle_between_ or test_plane_uses")

### Ex R9.18 — La droite qui passe par deux points, et les radians · `line_through`, `deg_to_rad` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer une équation cartésienne de la droite qui passe par deux points, verticales comprises, et la conversion en radians.  
**Prérequis :** fiche §4 et §6 · **mymath :** `plane.py` · **Parcours :** C

Écris `line_through(A, B)` puis `deg_to_rad(degrees)` (fiche §4 et §6). `line_through` renvoie un triplet $(a, b, c)$ tel que $ax + by + c = 0$ pour les deux points, avec $(a, b) \ne (0, 0)$, des entiers quand les coordonnées sont entières, et lève une `ValueError` si les deux points sont égaux. Méthode : un vecteur directeur $(d_x ; d_y)$, tourné d'un quart de tour, donne un vecteur normal $(-d_y ; d_x)$, soit $(a ; b)$ ; puis $c$ s'obtient en écrivant que $A$ est sur la droite.

Une droite a une infinité d'équations : la vérification les compare toutes sous une même forme, avec `normalized` (de la cellule des outils), qui divise $a$, $b$ et $c$ par le premier nombre non nul parmi $a$ et $b$.

La vérification calcule :
- a) `normalized(line_through((1, -1), (3, 5)))` ;
- b) `normalized(line_through((4, 1), (4, -7)))`, une droite verticale ;
- c) le nom de l'erreur levée par `line_through((2, 3), (2, 3))` ;
- d) le type du coefficient $a$ de `line_through((0, 1), (1, 3))`, en texte ;
- e) la liste `[deg_to_rad(d) for d in (180, 90, 45, -30)]` (4 décimales).

Puis, dans ta copie : pourquoi la méthode du quart de tour marche-t-elle aussi pour une droite verticale, alors que la pente n'existe pas ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R9")  # reload your saved file

In [ ]:
with checked("R9.18a"):
    wbm.check("R9.18a", normalized(mymath.plane.line_through((1, -1), (3, 5))), computed=True)
with checked("R9.18b"):
    wbm.check("R9.18b", normalized(mymath.plane.line_through((4, 1), (4, -7))), computed=True)
with checked("R9.18c"):
    wbm.check("R9.18c", raised(mymath.plane.line_through, (2, 3), (2, 3)), computed=True)
with checked("R9.18d"):
    wbm.check("R9.18d", type(mymath.plane.line_through((0, 1), (1, 3))[0]).__name__, computed=True)
with checked("R9.18e"):
    wbm.check("R9.18e", [mymath.plane.deg_to_rad(d) for d in (180, 90, 45, -30)], computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r9_plane.py", "test_line_through_ or test_deg_to_rad_ or test_plane_uses")

### Ex R9.19 — Les mêmes calculs avec NumPy (`np.linalg.norm`, `@`) 📦 ★ ⏱️ 15 min
**Objectif :** refaire avec NumPy les calculs de norme, de produit scalaire, de distance et d'angle.  
**Prérequis :** fiche §3, §5, §7 et §8 ; workbookIA 0A §100.8.1 et §100.8.3 · **Parcours :** C, E

La cellule fournie définit deux vecteurs de $\mathbb{R}^4$, sous forme d'arrays de NumPy. Calcule, **avec NumPy** (`np.linalg.norm`, l'opérateur `@`, `np.arccos`, `np.degrees`), sans écrire de boucle :
- a) la norme de `x` ;
- b) le produit scalaire de `x` et `y` ;
- c) la norme L1 de `x` (`ord=1`) ;
- d) la norme L∞ de `x` (`ord=np.inf`) ;
- e) la similarité cosinus de `x` et `y` (4 décimales) ;
- f) la distance entre `x` et `y` (4 décimales) ;
- g) l'angle entre `x` et `y`, en degrés (2 décimales).

Puis vérifie que tes fonctions `norm` et `dot` de `mymath` donnent les mêmes valeurs que NumPy sur `x` et `y` (passe-leur `tuple(x)`). Dans ta copie : l'angle de g) est-il aigu ou obtus ? Comment le savoir dès b) ?

In [ ]:
x = np.array([2.0, -1.0, -4.0, 2.0])
y = np.array([1.0, 3.0, 1.0, 0.5])

In [ ]:
answer_R9_19a = ...  # a) np.linalg.norm(...)
answer_R9_19b = ...  # b) with @
answer_R9_19c = ...  # c) ord=1
answer_R9_19d = ...  # d) ord=np.inf
answer_R9_19e = ...  # e) the cosine similarity
answer_R9_19f = ...  # f) the distance
answer_R9_19g = ...  # g) the angle in degrees

In [ ]:
for letter, answer in zip("abcdefg", [answer_R9_19a, answer_R9_19b, answer_R9_19c, answer_R9_19d,
                                       answer_R9_19e, answer_R9_19f, answer_R9_19g]):
    wbm.check(f"R9.19{letter}", answer)

**Comparer avec ta librairie** (la cellule s'exécute aussi tant que `mymath` n'est pas écrite).

In [ ]:
with checked("R9.19 mymath"):
    same_norm = math.isclose(mymath.plane.norm(tuple(x)), float(np.linalg.norm(x)))
    same_dot = math.isclose(mymath.plane.dot(tuple(x), tuple(y)), float(x @ y))
    verdict("R9.19 mymath", same_norm and same_dot, "norm et dot donnent les mêmes valeurs que NumPy.",
            "norm ou dot ne donne pas la même valeur que NumPy : relance leurs tests (R9.16, R9.17).")

## Partie B · Une prédiction, une figure et une expérience

Fiche §2, §6, §7 et §8. Ce qui change, ou non, la similarité cosinus de deux vecteurs, deux vecteurs et leur somme en image, puis le cosinus et le sinus sur un tour complet.

### Ex R9.20 — Prédire : permuter, décaler ou retourner des coordonnées change-t-il la similarité cosinus ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir l'effet d'un changement d'ordre des coordonnées, d'un décalage de toutes les coordonnées et d'un changement de signe d'une seule coordonnée sur la similarité cosinus.  
**Prérequis :** fiche §5, §7 et §8 · **Parcours :** C, E

Les vecteurs $\mathbf{u} = (3, -1, 4, 1, -5)$ et $\mathbf{v} = (2, 2, -1, 3, 0)$ ont cinq coordonnées, et une similarité cosinus proche de 0. **Avant d'exécuter quoi que ce soit**, prédis :
- a) la similarité cosinus des deux vecteurs obtenus en écrivant les coordonnées de $\mathbf{u}$ et celles de $\mathbf{v}$ dans l'ordre inverse, $(-5, 1, 4, -1, 3)$ et $(0, 3, -1, 2, 2)$ : (1) l'opposée ; (2) la même ; (3) une autre valeur ;
- b) la similarité cosinus des deux vecteurs obtenus en ajoutant 10 à **chaque** coordonnée de $\mathbf{u}$ et de $\mathbf{v}$ : (1) plus petite ; (2) la même ; (3) plus proche de 1 ;
- c) la similarité cosinus de $(-3, -1, 4, 1, -5)$, où seule la **première** coordonnée de $\mathbf{u}$ a changé de signe, et de $\mathbf{v}$ : (1) l'opposée ; (2) la même ; (3) une autre valeur.

Réponds par le numéro du choix : 1, 2 ou 3. Écris tes trois prédictions, vérifie-les, puis lance l'expérience, qui calcule ces similarités avec NumPy.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R9_20a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_R9_20b = ...  # b) 1, 2 or 3
prediction_R9_20c = ...  # c) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abc", [prediction_R9_20a, prediction_R9_20b, prediction_R9_20c]):
    wbm.check(f"R9.20{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes trois prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R9_20a, prediction_R9_20b, prediction_R9_20c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    u = np.array([3, -1, 4, 1, -5])
    v = np.array([2, 2, -1, 3, 0])
    reverse = [4, 3, 2, 1, 0]                  # the coordinates in the reverse order
    flip = np.array([-1, 1, 1, 1, 1])          # the sign of the first coordinate changed


    def cosine(a, b):
        return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


    print("   cos(u, v)                   =", round(cosine(u, v), 4))
    print("a) cos(u[reverse], v[reverse]) =", round(cosine(u[reverse], v[reverse]), 4))
    print("b) cos(u + 10, v + 10)         =", round(cosine(u + 10, v + 10), 4))
    print("c) cos(flip * u, v)            =", round(cosine(flip * u, v), 4))

**Après l'expérience**, dans ta copie : explique a) et c) en écrivant le produit scalaire et les normes comme des sommes de cinq termes : lesquels changent ? Que faudrait-il changer dans $\mathbf{u}$ pour obtenir l'opposée de la similarité ? Pour b), compare les directions de $\mathbf{u} + 10$ et de $\mathbf{v} + 10$ à celle du vecteur $(1, 1, 1, 1, 1)$ : pourquoi ajouter un même nombre très grand à toutes les coordonnées rapprocherait-il la similarité cosinus de 1 ?

### Ex R9.21 — Dessiner des vecteurs, leur somme et un angle 🎨 ★★ ⏱️ 20 min
**Objectif :** dessiner deux vecteurs, leur somme et la valeur de leur angle, dans des axes de même unité.  
**Prérequis :** fiche §2 et §7 ; R0 (matplotlib) · **Parcours :** C, E

Complète `draw_vectors(ax, u, v)`, qui dessine dans l'axe `ax` qu'elle reçoit :
- trois flèches parties de l'origine, $\vec{u}$, $\vec{v}$ et $\vec{u} + \vec{v}$, avec `ax.quiver(xs, ys, us, vs, angles="xy", scale_units="xy", scale=1)` : `xs` et `ys` sont les points de départ (ici des 0), `us` et `vs` les coordonnées des vecteurs, et les trois derniers réglages font que chaque flèche a exactement la longueur de son vecteur ;
- en pointillés, les deux autres côtés du parallélogramme (facultatif) ;
- la même unité sur les deux axes (`ax.set_aspect("equal")`), des limites qui laissent voir les trois flèches, et une grille ;
- un titre qui donne l'angle entre $\vec{u}$ et $\vec{v}$, en degrés, avec 1 décimale, **calculé dans la fonction** (par exemple `f"angle = {angle:.1f}°"`).

La cellule suivante dessine $\vec{u} = (5 ; 1)$ et $\vec{v} = (-2 ; 3)$ ; une autre la vérifie, sur ces vecteurs et sur une autre paire.

Dans ta copie : sans la ligne `ax.set_aspect("equal")`, l'angle **dessiné** a-t-il encore l'air de valoir celui du titre ? Pourquoi ?

In [ ]:
def draw_vectors(ax, u, v):
    """Draw u, v and u + v from the origin in ax (ax.quiver), with equal axes and the angle in the title."""
    raise NotImplementedError("ax.quiver pour les trois flèches, ax.set_aspect, puis le titre avec l'angle")

In [ ]:
with checked("R9.21 figure"):
    fig, ax = plt.subplots(figsize=(5, 5))
    try:
        created = drawn(draw_vectors, ax, (5, 1), (-2, 3))
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex R9.21 : ta fonction ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans l'axe "
              "qu'elle reçoit.")
    else:
        plt.show()

**Vérifications automatiques** (la figure elle-même se juge à l'œil).

In [ ]:
with checked("R9.21 vérifications"):
    problems = []
    for u, v, angle in [((5, 1), (-2, 3), "112.4"), ((-3, 2), (1, 4), "70.3")]:
        fig, ax = plt.subplots(figsize=(5, 5))
        try:
            created = drawn(draw_vectors, ax, u, v)
        finally:
            plt.close(fig)                                 # only the checks below are shown here
        if created:
            problems.append("ta fonction ouvre sa propre figure : dessine dans l'axe reçu (ax.quiver, …)")
            break
        arrows = set()
        quivers = [q for q in ax.collections if hasattr(q, "U") and hasattr(q, "V")]
        for q in quivers:
            for a, b in zip(np.ravel(q.U), np.ravel(q.V)):
                arrows.add((round(float(a), 6), round(float(b), 6)))
        wanted = {tuple(float(c) for c in u), tuple(float(c) for c in v),
                  tuple(float(a + b) for a, b in zip(u, v))}
        if not wanted <= arrows:
            problems.append("trois flèches avec ax.quiver, parties de l'origine : u, v et u + v")
        elif any(getattr(q, "angles", None) != "xy" or getattr(q, "scale_units", None) != "xy"
                 or getattr(q, "scale", None) != 1 for q in quivers):
            problems.append("les réglages angles=\"xy\", scale_units=\"xy\", scale=1 de ax.quiver, pour que chaque "
                            "flèche ait exactement la longueur de son vecteur")
        x0, x1 = sorted(ax.get_xlim())
        y0, y1 = sorted(ax.get_ylim())
        tips = [(0, 0), u, v, (u[0] + v[0], u[1] + v[1])]
        if not all(x0 <= px <= x1 and y0 <= py <= y1 for px, py in tips):
            problems.append("des limites qui laissent voir les trois flèches en entier (ax.set_xlim, ax.set_ylim)")
        if ax.get_aspect() not in (1, 1.0, "equal"):
            problems.append("la même unité sur les deux axes : ax.set_aspect(\"equal\")")
        title = ax.get_title().replace(",", ".")
        if angle not in title:
            problems.append("l'angle entre u et v dans le titre, en degrés, avec 1 décimale (calculé dans la "
                            "fonction)")
        if problems:
            break
    if problems:
        print("❌ Ex R9.21 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R9.21 : les trois flèches, la même unité sur les deux axes et l'angle dans le titre.")

### Ex R9.22 — Le cercle trigonométrique : cos et sin d'un angle qui tourne 🔬 ★★ ⏱️ 20 min
**Objectif :** explorer numériquement le cosinus et le sinus sur un tour complet : la relation cos² + sin² = 1, leurs comparaisons, leur maximum et la dérivée du sinus.  
**Prérequis :** fiche §6 ; workbookIA 0A §100.8.3 (calcul vectorisé) · **Parcours :** C

La cellule fournie définit `degrees`, 360 angles en degrés (0,5°, 1,5°, …, 359,5°), et `whole`, les angles entiers de 0° à 359°. Avec NumPy (`np.radians`, `np.cos`, `np.sin`, `np.max`, `np.abs`, `np.sum`, `np.argmax`) :
- a) `close_to_1` : vrai si, pour tous les angles de `degrees`, $\lvert \cos^2\theta + \sin^2\theta - 1 \rvert$ reste inférieur à $10^{-12}$ (un booléen) ;
- b) `first_sin_above` : le premier angle de `degrees`, en degrés, pour lequel $\sin\theta > \cos\theta$ (sur un tableau de booléens, `np.argmax` donne la position du premier `True`) ;
- c) `argmax_sin` : l'angle de `whole`, en degrés, pour lequel $\sin\theta$ est le plus grand ;
- d) `slope` : le taux d'accroissement $\frac{\sin(1 + h) - \sin 1}{h}$, avec $h = 10^{-5}$ (6 décimales), à comparer avec $\cos 1$.

Puis, dans la cellule qui suit la vérification, trace sur un même graphique $\cos$ et $\sin$ entre 0 et $4\pi$, avec une légende et l'abscisse en radians.

Dans ta copie : sur quels intervalles d'angles a-t-on $\cos\theta > \sin\theta$ ? Dessine-les sur le cercle. Que montre d) sur la dérivée de $\sin$ ?

In [ ]:
degrees = np.arange(360) + 0.5          # 0.5°, 1.5°, ..., 359.5°: no angle where cos = sin exactly
whole = np.arange(360)                   # 0°, 1°, ..., 359°

In [ ]:
close_to_1 = ...       # a) a boolean
first_sin_above = ...  # b) an angle in degrees
argmax_sin = ...       # c) an angle in degrees
slope = ...            # d) a number

In [ ]:
wbm.check("R9.22a", close_to_1)
wbm.check("R9.22b", first_sin_above)
wbm.check("R9.22c", argmax_sin)
wbm.check("R9.22d", slope)

**Le tracé** de $\cos$ et $\sin$ entre 0 et $4\pi$ (`np.linspace`, `ax.plot`, `ax.legend`).

In [ ]:
# the plot of cos and sin on [0, 4π], with a legend and the angle in radians on the x-axis

## Partie C · Radians, degrés et flottants

Fiche §5 et §6. Trois confusions entre degrés et radians à trouver, puis un test d'orthogonalité qui résiste aux arrondis.

### Ex R9.23 — Chasse à l'erreur : radians et degrés 🐛 ★ ⏱️ 15 min
**Objectif :** repérer les confusions entre degrés et radians dans du code, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §6 · **Parcours :** C, E

Voici cinq lignes de code, chacune avec ce qu'en attend son auteur. Trois sont fausses. **Sans exécuter d'abord**, trouve-les.

- **(A)** « Le sinus de 45° : `math.sin(45)`. »
- **(B)** « Le sinus de 45° : `math.sin(math.radians(45))`. »
- **(C)** « $\frac{\pi}{3}$ radian, converti en degrés : `(math.pi / 3) * 180 / math.pi`. »
- **(D)** « L'angle, en degrés, dont le cosinus vaut 0,5 : `math.acos(0.5)`. »
- **(E)** « 30° en radians : `30 * 180 / math.pi`. »

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des cinq lignes, de (A) à (E), la bonne valeur de ce que décrit la phrase de son auteur (une liste de 5 nombres, 4 décimales).

Tu peux exécuter les lignes après avoir répondu, pour vérifier : une ligne juste peut afficher un résultat comme `59.99999999999999`, un arrondi des flottants (R0), pas une erreur.

Dans ta copie : pour chaque ligne fausse, quelle règle a été oubliée ?

In [ ]:
answer_R9_23a = ...  # a) letters, as text
answer_R9_23b = ...  # b) a list of 5 values (4 decimals)

In [ ]:
wbm.check("R9.23a", answer_R9_23a)
wbm.check("R9.23b", answer_R9_23b)

### Ex R9.24 — Comparer deux flottants : `np.isclose` 🛠️ ★ ⏱️ 10 min
**Objectif :** tester l'orthogonalité de vecteurs à coordonnées décimales sans être trompé par les arrondis.  
**Prérequis :** fiche §5 ; R0 (les flottants) · **Parcours :** C

Sur papier, $\vec{u} = (0{,}1 ; 0{,}7)$ et $\vec{v} = (2{,}1 ; -0{,}3)$ sont orthogonaux : $0{,}21 - 0{,}21 = 0$. La cellule fournie calcule leur produit scalaire en flottants : il ne vaut pas exactement 0 (R0 : les flottants forment une grille). Un test `u·v == 0` se tromperait donc.

1. Écris `is_orthogonal(u, v, tol=1e-9)`, qui renvoie `True` quand le produit scalaire est nul **à `tol` près**, en valeur absolue. Deux outils font cette comparaison : `math.isclose(x, 0, abs_tol=tol)` et `np.isclose(x, 0, atol=tol)` (qui renvoie un booléen de NumPy : convertis-le avec `bool(...)`).
2. La cellule de vérification essaie ta fonction sur cinq cas.

Dans ta copie : que renvoie `math.isclose(1e-17, 0)`, sans `abs_tol` ? Pourquoi une tolérance **relative** (« à 1e-9 près de la valeur ») ne peut-elle pas servir pour comparer à 0 ?

In [ ]:
u = (0.1, 0.7)
v = (2.1, -0.3)                          # 0.1 × 2.1 − 0.7 × 0.3 = 0.21 − 0.21 = 0: orthogonal on paper
print("u·v =", u[0] * v[0] + u[1] * v[1])

In [ ]:
def is_orthogonal(u, v, tol=1e-9):
    """True when u·v is 0 up to tol (in absolute value): a float dot product is rarely exactly 0."""
    raise NotImplementedError("le produit scalaire, puis math.isclose(..., 0, abs_tol=tol)")

In [ ]:
with checked("R9.24 vérifications"):
    problems = []
    result = is_orthogonal(u, v)
    if result is None:
        problems.append("is_orthogonal ne renvoie rien (None) : il manque un return (print affiche, mais ne renvoie "
                        "pas)")
    elif not isinstance(result, bool):
        kind = ("un booléen de NumPy (np.isclose en renvoie un)" if type(result).__module__ == "numpy"
                else f"un objet de type {type(result).__name__}")
        problems.append(f"is_orthogonal doit renvoyer un booléen de Python, True ou False, et non {kind} : "
                        "convertis avec bool(...)")
    elif result is not True:
        problems.append("is_orthogonal(u, v) doit être True : sur papier, u·v vaut 0")
    elif is_orthogonal((1, 2), (-3, 1)) is not False:
        problems.append("is_orthogonal((1, 2), (-3, 1)) doit être False : u·v vaut −1 ; compare sa valeur absolue "
                        "à tol")
    if not problems and is_orthogonal((1, 0), (1e-6, 1)) is not False:
        problems.append("is_orthogonal((1, 0), (1e-6, 1)) doit être False : 1e-6 dépasse la tolérance")
    if not problems and is_orthogonal((3, 4), (-4, 3)) is not True:
        problems.append("is_orthogonal((3, 4), (-4, 3)) doit être True")
    if not problems and is_orthogonal((2, 1), (1, 1), tol=10) is not True:
        problems.append("le paramètre tol doit être utilisé : avec tol=10, (2 ; 1) et (1 ; 1) passent")
    if problems:
        print("❌ Ex R9.24 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R9.24 : is_orthogonal tolère les arrondis des flottants, et seulement eux.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Calcules-tu une distance, un vecteur, une norme et l'équation d'une droite ?
2. Sais-tu calculer un produit scalaire, reconnaître des vecteurs orthogonaux et en déduire un angle ?
3. Passes-tu des degrés aux radians, et lis-tu cos et sin sur le cercle trigonométrique ?

**La suite** : l'examen blanc CP0, le bilan de la remise à niveau. Puis, dans workbookIA, le chapitre 0B.